## SAT Loggbok

tolkade news alert till Wikidata platser
* denna notebook [387_SATLoggbok.ipynb](https://github.com/salgo60/Stockholm_Archipelago_Trail/tree/main/Notebook/387_SATLoggbok.ipynb)
* Issue [#387](https://github.com/salgo60/Stockholm_Archipelago_Trail/issues/387)

Version 
* 1.0 created
* 1.1 added more languages and geojson
* 1.2 add social accounts, webpage; bild
   * OSMID numeriskt
   * type bort blir services["sale", "stamp"]
   * nytt namn sat-passport-service-points


In [1]:
import time
import datetime  
start_time = time.time()
start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Started: {start_str}")


Started: 2026-06-04 22:07


In [2]:
import requests
import json
from datetime import datetime, UTC

year = "2026"
version = "1_2"

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"

query = r"""
SELECT
  ?place
  ?placeLabel
  ?coord
  ?SATpart
  ?SATpartLabel
  ?type
  ?osmNode
  ?osmWay
  ?osmRelation 
  ?website
  ?image
  ?instanceOf
  ?instanceOfLabel
  ?facebook
  ?facebookId
  ?instagram
  ?label_sv
  ?label_en
  ?label_de
  ?label_fr
  ?label_ar 
  ?label_es 
  ?label_nn 
  ?label_da 
  ?label_fi 
  ?label_ru 
  ?label_it
  ?label_zh
  
 
WHERE {

  VALUES (?place ?type) {
    (wd:Q134007891 "sale_and_stamp")
    (wd:Q134676671 "sale_and_stamp")
    (wd:Q18291743  "sale_and_stamp")
    (wd:Q134006996 "sale_and_stamp")
    (wd:Q133829736 "sale_and_stamp")
    (wd:Q136337707 "sale_and_stamp")
    (wd:Q133812552 "sale_and_stamp")
    (wd:Q133812682 "sale_and_stamp")
    (wd:Q134705878 "sale_and_stamp")
    (wd:Q134571436 "sale_and_stamp")
    (wd:Q133502889 "sale_and_stamp")
    (wd:Q134572952 "sale_and_stamp")
    (wd:Q134473761 "sale_and_stamp")
    (wd:Q15695240  "sale_and_stamp")
    (wd:Q134529162 "sale_and_stamp")
    (wd:Q133840505 "sale_and_stamp")
    (wd:Q140053969 "sale_and_stamp")
    (wd:Q134589221 "sale_and_stamp")
    (wd:Q134311495 "sale_and_stamp")
    (wd:Q135437122 "sale_and_stamp")
    (wd:Q140054528 "sale_and_stamp")

    (wd:Q134523792 "stamp_only")
    (wd:Q134705637 "stamp_only")
    (wd:Q134581015 "stamp_only")
    (wd:Q134311974 "stamp_only")
    (wd:Q134589066 "stamp_only")
    (wd:Q134311548 "stamp_only")
    (wd:Q134587974 "stamp_only")
    (wd:Q134776972 "stamp_only")
    (wd:Q134306927 "stamp_only")
    (wd:Q135003463 "stamp_only")
    (wd:Q134527248 "stamp_only")
  }
  OPTIONAL {
      ?place wdt:P31 ?instanceOf .
    }

  OPTIONAL {
    ?place wdt:P2789 ?SATpart .
    ?SATpart wdt:P361 wd:Q131318799 .
  }
  OPTIONAL { ?place wdt:P11693 ?osmNode }
  OPTIONAL { ?place wdt:P10689 ?osmWay }
  OPTIONAL { ?place wdt:P402 ?osmRelation }  

  OPTIONAL { ?place wdt:P856 ?website }
  OPTIONAL { ?place wdt:P18 ?image }
    
  OPTIONAL { ?place wdt:P2013 ?facebook }
  OPTIONAL { ?place wdt:P11705 ?facebookId }
    
  OPTIONAL { ?place wdt:P2003 ?instagram }

  OPTIONAL { ?place rdfs:label ?label_sv FILTER(LANG(?label_sv)="sv") }
  OPTIONAL { ?place rdfs:label ?label_en FILTER(LANG(?label_en)="en") }
  OPTIONAL { ?place rdfs:label ?label_de FILTER(LANG(?label_de)="de") }
  OPTIONAL { ?place rdfs:label ?label_fr FILTER(LANG(?label_fr)="fr") }
  OPTIONAL { ?place rdfs:label ?label_ar FILTER(LANG(?label_ar)="ar") }
  OPTIONAL { ?place rdfs:label ?label_es FILTER(LANG(?label_es)="es") }
  OPTIONAL { ?place rdfs:label ?label_nn FILTER(LANG(?label_nn)="nn") }
  OPTIONAL { ?place rdfs:label ?label_da FILTER(LANG(?label_da)="da") }
  OPTIONAL { ?place rdfs:label ?label_fi FILTER(LANG(?label_fi)="fi") }
  OPTIONAL { ?place rdfs:label ?label_ru FILTER(LANG(?label_ru)="ru") }
  OPTIONAL { ?place rdfs:label ?label_it FILTER(LANG(?label_it)="it") }
  OPTIONAL { ?place rdfs:label ?label_zh FILTER(LANG(?label_zh)="zh") }
  ?place wdt:P625 ?coord .

  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "sv,en".
  }
}
ORDER BY ?placeLabel
"""


response = requests.get(
    SPARQL_ENDPOINT,
    params={
        "query": query,
        "format": "json"
    },
    headers={
        "User-Agent": "SAT-Passport-Exporter/1.0 salgo60@msn.com"
    }
)

response.raise_for_status()

data = response.json()


def parse_wkt_point(wkt):
    """
    Point(18.12345 59.12345)
    -> lat, lon
    """
    wkt = wkt.replace("Point(", "").replace(")", "")
    lon, lat = map(float, wkt.split())
    return lat, lon


locations = {}

for row in data["results"]["bindings"]:

    qid = row["place"]["value"].split("/")[-1]

    #
    # Create location only once
    #
    if qid not in locations:

        names = {}

        for lang in [
            "sv", "en", "de", "fr",
            "ar", "es", "nn", "da",
            "fi", "ru", "it", "zh"
        ]:
            key = f"label_{lang}"

            if key in row:
                names[lang] = row[key]["value"]

        lat, lon = parse_wkt_point(
            row["coord"]["value"]
        )
        location = {
            "wikidata": qid,
            "wikidata_url": f"https://www.wikidata.org/wiki/{qid}",
            "name": row["placeLabel"]["value"],
            "names": names,
            "latitude": lat,
            "longitude": lon,
            "instance_of": []
        }
        if row["type"]["value"] == "sale_and_stamp":
            location["passport_services"] = [
                "sale",
                "stamp"
            ]
        else:
            location["passport_services"] = [
                "stamp"
            ]

        if "image" in row:

            image_name = row["image"]["value"]
        
            location["image_url"] = image_name
        
        #
        # SAT section
        #
        if "SATpart" in row:
            sat_qid = row["SATpart"]["value"].split("/")[-1]

            location["sat_part"] = {
                "wikidata": sat_qid,
                "wikidata_url": f"https://www.wikidata.org/wiki/{sat_qid}",
                "label": row["SATpartLabel"]["value"]
            }
            
        #
        # Website
        #
        if "website" in row:
            location["website"] = row["website"]["value"]

        #
        # Social media
        #
        social = {}
        
        if "facebook" in row:
            social["facebook_username"] = row["facebook"]["value"]
        
        if "facebookId" in row:
            social["facebook_id"] = row["facebookId"]["value"]
        
        if "instagram" in row:
            social["instagram_username"] = row["instagram"]["value"]

        if social:
            location["social"] = social

        #
        # OSM
        # 

        if "osmNode" in row:
            location["osm"] = {
                "type": "node",
                "id": int(row["osmNode"]["value"])
            }

        elif "osmWay" in row:
            location["osm"] = {
                "type": "way",
                "id": int(row["osmWay"]["value"])
            }

        elif "osmRelation" in row:
            location["osm"] = {
                "type": "relation",
                "id": int(row["osmRelation"]["value"])
            }
        if "osm" in location:
            location["osm_url"] = (
                f"https://www.openstreetmap.org/"
                f"{location['osm']['type']}/"
                f"{location['osm']['id']}"
            )
        locations[qid] = location

    #
    # Handle multiple P31 values
    #
    if (
        "instanceOf" in row and
        "instanceOfLabel" in row
    ):

        instance_qid = row["instanceOf"]["value"].split("/")[-1]
        instance = {
            "wikidata": instance_qid,
            "wikidata_url": f"https://www.wikidata.org/wiki/{instance_qid}",
            "label": row["instanceOfLabel"]["value"]
        }
        

        if instance not in locations[qid]["instance_of"]:
            locations[qid]["instance_of"].append(instance)

#
# Convert dict -> list
#
locations = sorted(
    locations.values(),
    key=lambda x: x["name"]
)


export = {
    "dataset": "sat-passport-service-points",
    "export_version": f"{year}.{version}",
    "schema_version": "1.0",
    "generated": datetime.now(UTC).isoformat(),
    "source_system": "Wikidata",
    "source_license": "CC0-1.0",
    "query_service": "https://query.wikidata.org/",
    "generated_by": "SAT Passport Notebook",
    "location_count": len(locations),
    "sale_and_stamp_count": sum(
        1
        for loc in locations
        if "sale" in loc["passport_services"]
    ),

    "stamp_only_count": sum(
        1
        for loc in locations
        if loc["passport_services"] == ["stamp"]
    ),
    "locations": locations
}

json_filename = (
    f"sat-passport-service-points-{year}-{version}.json"
)

with open(
    json_filename,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        export,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    f"Created {json_filename} "
    f"with {len(locations)} locations"
)

print(
    json.dumps(
        locations[:3],
        indent=2,
        ensure_ascii=False
    )
)


Created sat-passport-service-points-2026-1_2.json with 32 locations
[
  {
    "wikidata": "Q134676671",
    "wikidata_url": "https://www.wikidata.org/wiki/Q134676671",
    "name": "Arholma Nord",
    "names": {
      "sv": "Arholma Nord",
      "en": "Arholma Nord",
      "de": "Arholma Nord",
      "fr": "Arholma Nord",
      "ar": "أرهولما نورد",
      "es": "Arholma Nord",
      "da": "Arholma Nord",
      "fi": "Arholma Nord",
      "ru": "Архольма Норд",
      "it": "Arholma Nord",
      "zh": "Arholma Nord"
    },
    "latitude": 59.86079312,
    "longitude": 19.123592134,
    "instance_of": [
      {
        "wikidata": "Q27686",
        "wikidata_url": "https://www.wikidata.org/wiki/Q27686",
        "label": "hotell"
      },
      {
        "wikidata": "Q11707",
        "wikidata_url": "https://www.wikidata.org/wiki/Q11707",
        "label": "restaurang"
      }
    ],
    "passport_services": [
      "sale",
      "stamp"
    ],
    "image_url": "http://commons.wikimedia.org/

## Skapa geojson

In [3]:
# Create GeoJSON from locations

geojson = {
    "type": "FeatureCollection",
    "dataset": export["dataset"],
    "export_version": export["export_version"],
    "schema_version": export["schema_version"],
    "generated": export["generated"],
    "source_system": export["source_system"],
    "source_license": export["source_license"],
    "location_count": export["location_count"],
    "sale_and_stamp_count": export["sale_and_stamp_count"],
    "stamp_only_count": export["stamp_only_count"],
    "features": []
}

for location in locations:

    properties = {
        "wikidata": location["wikidata"],
        "wikidata_url": location["wikidata_url"],
        "name": location["name"],
        "names": location.get("names", {}),
        "passport_services": location.get(
            "passport_services",
            []
        ),
        "instance_of": location.get(
            "instance_of",
            []
        )
    }

    if "sat_part" in location:
        properties["sat_part"] = location["sat_part"]

    if "website" in location:
        properties["website"] = location["website"]

    if "image" in location:
        properties["image"] = location["image"]


    if "social" in location:
        properties["social"] = location["social"]

    if "osm" in location:
        properties["osm"] = location["osm"]

    if "osm_url" in location:
        properties["osm_url"] = location["osm_url"]

    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [
                location["longitude"],
                location["latitude"]
            ]
        },
        "properties": properties
    }

    geojson["features"].append(feature)

geojson_filename = (
    f"sat-passport-service-points-{year}-{version}.geojson"
)

with open(
    geojson_filename,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        geojson,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    f"Created {geojson_filename} "
    f"with {len(geojson['features'])} features"
)

print(
    json.dumps(
        geojson["features"][0],
        indent=2,
        ensure_ascii=False
    )
)

Created sat-passport-service-points-2026-1_2.geojson with 32 features
{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [
      19.123592134,
      59.86079312
    ]
  },
  "properties": {
    "wikidata": "Q134676671",
    "wikidata_url": "https://www.wikidata.org/wiki/Q134676671",
    "name": "Arholma Nord",
    "names": {
      "sv": "Arholma Nord",
      "en": "Arholma Nord",
      "de": "Arholma Nord",
      "fr": "Arholma Nord",
      "ar": "أرهولما نورد",
      "es": "Arholma Nord",
      "da": "Arholma Nord",
      "fi": "Arholma Nord",
      "ru": "Архольма Норд",
      "it": "Arholma Nord",
      "zh": "Arholma Nord"
    },
    "passport_services": [
      "sale",
      "stamp"
    ],
    "instance_of": [
      {
        "wikidata": "Q27686",
        "wikidata_url": "https://www.wikidata.org/wiki/Q27686",
        "label": "hotell"
      },
      {
        "wikidata": "Q11707",
        "wikidata_url": "https://www.wikidata.org/wiki/Q11707",
        

In [4]:
end_time = time.time()
duration = end_time - start_time

print(f"Finished in {duration:.2f} seconds.")


Finished in 4.75 seconds.
